# Voice Similarity Checker

Determines whether two voice recordings belong to the same person using **MFCC cosine distance** and spectral analysis.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DaniAngel79/voice-similarity-checker/blob/main/voice_similarity_checker.ipynb)

---
## How it works
1. Load two audio files (WAV or MP3)
2. Extract 13 MFCC coefficients per audio (mean over time)
3. Compute **cosine distance** between MFCC vectors
4. Extract spectral features: centroid, flatness, rolloff
5. Verdict: distance < 0.15 → same person
6. Save spectrogram image + PDF report

In [ ]:
# Install dependencies
!pip install librosa gradio fpdf2 soundfile -q

In [ ]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
from fpdf import FPDF
import gradio as gr


def load_audio(audio_path):
    """Load WAV or MP3. Returns (signal, sample_rate)."""
    audio, sr = librosa.load(audio_path, sr=None)
    return audio, sr


def calculate_spectral_features(audio, sr):
    """Extract MFCC (13 coeff) + spectral centroid, flatness, rolloff."""
    S = np.abs(librosa.stft(audio))
    spectral_centroid = librosa.feature.spectral_centroid(S=S)[0]
    spectral_flatness = librosa.feature.spectral_flatness(S=S)[0]
    spectral_rolloff  = librosa.feature.spectral_rolloff(S=S)[0]
    mfcc      = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    return spectral_centroid, spectral_flatness, spectral_rolloff, mfcc_mean


def compare_audio_features(f1, f2):
    """Return spectral score, MFCC cosine distance and per-feature diffs."""
    min_c = min(len(f1[0]), len(f2[0]))
    min_f = min(len(f1[1]), len(f2[1]))
    min_r = min(len(f1[2]), len(f2[2]))
    centroid_diff = np.mean(np.abs(f1[0][:min_c] - f2[0][:min_c]))
    flatness_diff = np.mean(np.abs(f1[1][:min_f] - f2[1][:min_f]))
    rolloff_diff  = np.mean(np.abs(f1[2][:min_r] - f2[2][:min_r]))
    cosine_sim    = np.dot(f1[3], f2[3]) / (np.linalg.norm(f1[3]) * np.linalg.norm(f2[3]))
    mfcc_distance = 1 - cosine_sim
    spectral_score = centroid_diff + flatness_diff + rolloff_diff
    return spectral_score, mfcc_distance, centroid_diff, flatness_diff, rolloff_diff


def generate_pdf_report(path1, path2, spectral_score, mfcc_distance,
                         centroid_diff, flatness_diff, rolloff_diff, verdict):
    """Save a PDF report with all metrics."""
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", "B", 16)
    pdf.cell(0, 10, "Audio Similarity Report", ln=True, align="C")
    pdf.ln(5)
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, f"Audio 1: {path1}", ln=True)
    pdf.cell(0, 10, f"Audio 2: {path2}", ln=True)
    pdf.ln(5)
    pdf.set_font("Arial", "B", 12)
    pdf.cell(0, 10, "Results:", ln=True)
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, f"Verdict: {verdict}", ln=True)
    pdf.cell(0, 10, f"Spectral Score: {spectral_score:.4f}", ln=True)
    pdf.cell(0, 10, f"MFCC Cosine Distance: {mfcc_distance:.4f}  (< 0.15 = same person)", ln=True)
    pdf.cell(0, 10, f"Spectral Centroid diff: {centroid_diff:.4f}", ln=True)
    pdf.cell(0, 10, f"Spectral Flatness diff: {flatness_diff:.6f}", ln=True)
    pdf.cell(0, 10, f"Spectral Rolloff diff:  {rolloff_diff:.4f}", ln=True)
    pdf.output("similarity_report.pdf")
    print("PDF report saved as similarity_report.pdf")


def determine_similarity(audio_path_1, audio_path_2):
    """Compare two audio files. Returns (verdict, mfcc_distance, spectrogram_path)."""
    audio_1, sr_1 = load_audio(audio_path_1)
    audio_2, sr_2 = load_audio(audio_path_2)

    if sr_1 != sr_2:
        print(f"Warning: different sample rates ({sr_1} vs {sr_2} Hz).")

    f1 = calculate_spectral_features(audio_1, sr_1)
    f2 = calculate_spectral_features(audio_2, sr_2)

    spectral_score, mfcc_distance, centroid_diff, flatness_diff, rolloff_diff = \n        compare_audio_features(f1, f2)

    verdict = "Same person." if mfcc_distance < 0.15 else "Different persons."
    print(verdict)

    # Spectrograms
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    librosa.display.specshow(
        librosa.amplitude_to_db(np.abs(librosa.stft(audio_1)), ref=np.max),
        sr=sr_1, x_axis="time", y_axis="log", ax=axes[0])
    axes[0].set_title("Spectrogram — Audio 1")
    fig.colorbar(axes[0].collections[0], ax=axes[0], format="%+2.0f dB")
    librosa.display.specshow(
        librosa.amplitude_to_db(np.abs(librosa.stft(audio_2)), ref=np.max),
        sr=sr_2, x_axis="time", y_axis="log", ax=axes[1])
    axes[1].set_title("Spectrogram — Audio 2")
    fig.colorbar(axes[1].collections[0], ax=axes[1], format="%+2.0f dB")
    plt.tight_layout()
    spectrogram_path = "spectrogram_output.png"
    plt.savefig(spectrogram_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Spectrogram saved as {spectrogram_path}")

    print(f"
MFCC Distance:          {mfcc_distance:.4f}  (threshold 0.15)")
    print(f"Spectral Centroid diff: {centroid_diff:.4f}")
    print(f"Spectral Flatness diff: {flatness_diff:.6f}")
    print(f"Spectral Rolloff diff:  {rolloff_diff:.4f}")

    generate_pdf_report(audio_path_1, audio_path_2, spectral_score, mfcc_distance,
                        centroid_diff, flatness_diff, rolloff_diff, verdict)
    return verdict, mfcc_distance, spectrogram_path

## Demo — Sample audio files
Runs with the two included WAV files. Expected output: spectrogram image + PDF report.

In [ ]:
# Run demo with included sample files
verdict, mfcc_dist, spec_path = determine_similarity("audio_path_1.wav", "audio_path_2.wav")
print(f"
Verdict      : {verdict}")
print(f"MFCC Distance: {mfcc_dist:.4f}")

## Interactive UI — Gradio
Launches a local web interface at .  
Upload **any two audio files** (WAV or MP3) to compare them interactively.

> On **Google Colab** a public share link is generated automatically when .

In [ ]:
def gradio_interface(audio1, audio2):
    if audio1 is None or audio2 is None:
        return "Please upload both audio files.", None
    verdict, mfcc_dist, spectrogram_path = determine_similarity(audio1, audio2)
    result_text = f"{verdict}
MFCC Distance: {mfcc_dist:.4f}"
    return result_text, spectrogram_path


interface = gr.Interface(
    fn=gradio_interface,
    inputs=[
        gr.Audio(type="filepath", label="Audio 1 (WAV or MP3)"),
        gr.Audio(type="filepath", label="Audio 2 (WAV or MP3)")
    ],
    outputs=[
        gr.Textbox(label="Result"),
        gr.Image(label="Spectrograms")
    ],
    title="Voice Similarity Checker",
    description=(
        "Upload two voice recordings to determine if they belong to the same person. "
        "Uses MFCC cosine distance + spectral analysis (centroid, flatness, rolloff)."
    ),
    examples=[
        ["audio_path_1.wav", "audio_path_2.wav"]
    ]
)

interface.launch(share=True)